# PF8 01webbuildings: buildings for the web page

Reads `data/buildings_santiago.parquet` from 00buildings and writes a compact file for the deck.gl page: `docs/data/buildings.json`.

Steps:
1. Drop building outlines that have parts drawn inside them, so towers such as Gran Torre Costanera are not drawn twice (the Simple 3D Buildings rule).
2. Give every building a height. Mapped heights stay as they are. Missing heights come from a type rule (houses 2 floors, garages and sheds 1 floor) or, for everything else, from the EU's GHS-BUILT-H grid (Average Net Building Height, 2018, 100 m cells, CC BY 4.0). The source is kept in `est`: 0 mapped, 1 type rule, 2 grid, 3 fallback.
3. Simplify outlines very slightly.
4. Find the ground height under every building from the same elevation tiles the web page uses (Mapzen Terrarium on AWS), so each building stands on the 3D terrain. (deck.gl's experimental TerrainExtension did not lift the buildings on Likren's PC, 19 Nov 2026, so the ground height is baked in here instead.)
5. Write one row per polygon: height, base height, estimate flag, name index, ground height, outline.

## 0. Settings

In [1]:
from pathlib import Path
import time
import pandas as pd

IN_FILE = Path("data") / "buildings_santiago.parquet"
OUT_DIR = Path("docs") / "data"
OUT_FILE = OUT_DIR / "buildings.json"

LEVEL_HEIGHT_M = 3.0  # <- metres per floor
HOUSE_FLOORS = 2  # <- floors for houses with no mapped height
SMALL_FLOORS = 1  # <- floors for garages, sheds, roofs and similar with no mapped height
FALLBACK_FLOORS = 2  # <- floors when neither a type rule nor the grid gives a height
HOUSE_TYPES = ["house", "semidetached_house", "detached", "bungalow", "terrace", "farm", "cabin", "static_caravan"]  # <- building types treated as houses
SMALL_TYPES = ["garage", "garages", "shed", "roof", "carport", "hut", "kiosk", "toilets", "greenhouse", "service", "transformer_tower", "guardhouse"]  # <- one-floor building types
GHS_TILES = ["R13_C11", "R12_C11", "R14_C11", "R13_C12", "R12_C12"]  # <- GHS-BUILT-H tiles tried in order until one covers the buildings
GHS_URL = "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_BUILT_H_GLOBE_R2023A/GHS_BUILT_H_ANBH_E2018_GLOBE_R2023A_4326_3ss/V1-0/tiles/GHS_BUILT_H_ANBH_E2018_GLOBE_R2023A_4326_3ss_V1_0_{tile}.zip"
GHS_DIR = Path("data") / "ghs_built_h"
MIN_GRID_HEIGHT_M = 3.0  # <- the grid's cell average is never allowed below one floor
SIMPLIFY_M = 0.3  # <- outline simplification tolerance in metres (0 = none)
DECIMALS = 6  # <- coordinate decimals (6 is about 0.1 m)

TERRAIN_ZOOM = 15  # <- zoom of the elevation tiles (15 is about 4 m per pixel here); match maxZoom in the web page
TERRAIN_URL = "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"  # <- same tiles as the web page
TERRAIN_DIR = Path("data") / f"terrarium{TERRAIN_ZOOM}"
SINK_M = 2.0  # <- VISUAL: sink each building this far into the ground so no gap shows under it on slopes

OUT_DIR.mkdir(parents=True, exist_ok=True)
TERRAIN_DIR.mkdir(parents=True, exist_ok=True)
GHS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Read

In [2]:
import geopandas as gpd

gdf = gpd.read_parquet(IN_FILE)
print(f"{len(gdf):,} rows")
print(gdf["kind"].value_counts().to_string())

71,874 rows
kind
building             71350
part                   512
building and part       12


## 2. Drop outlines that contain parts

A part belongs to an outline when the part's inside point falls inside the outline. Outlines with at least one part are dropped; the parts draw the building instead.

In [3]:
parts = gdf[gdf["kind"] != "building"]
outlines = gdf[gdf["kind"] == "building"]

part_points = gpd.GeoDataFrame(geometry=parts.representative_point(), crs=gdf.crs)
hits = gpd.sjoin(part_points, outlines[["geometry"]], predicate="within", how="inner")
outline_ids_with_parts = set(hits["index_right"])

keep = gdf.drop(index=list(outline_ids_with_parts))
print(f"Outlines with parts dropped: {len(outline_ids_with_parts):,}")
print(f"Rows kept: {len(keep):,}")

names_dropped = gdf.loc[list(outline_ids_with_parts), "name"].dropna().tolist()
print("Named outlines dropped:", names_dropped[:15])

Outlines with parts dropped: 131
Rows kept: 71,743
Named outlines dropped: ['Cenco Alto Las Condes', 'Correo Central', 'Mercado Central de Santiago', 'World Trade Center Santiago', 'Parque Arauco Oriente', 'Torre de La Costanera', 'Edificio Metrópolis', 'Facultad de Medicina UC', 'Casa Central Universidad Católica', 'Edificio del Pacífico', 'Palacio Falabella', 'Santander Arena', 'Boulevard Alameda 333', 'Palacio del Ex-Congreso Nacional de Chile', 'Centro de Alto Rendimiento']


## 3. Download the GHS-BUILT-H tile

GHS-BUILT-H R2023A gives the **average** height of the buildings in each 100 m cell (about 3 arc seconds), estimated from satellite elevation models and Sentinel-2 imagery for 2018. A cell average is right for blocks of similar buildings but pulls single towers down, so it is only used where OSM has no height.

The tile is downloaded once into the folder from Section 0. Tiles are tried in order until one covers the buildings.

In [4]:
import urllib.request
import zipfile
import rasterio
from rasterio.windows import from_bounds

west, south, east, north = keep.total_bounds
ghs_tif = None
for tile in GHS_TILES:
    zpath = GHS_DIR / f"ghs_anbh_{tile}.zip"
    if not zpath.exists():
        url = GHS_URL.format(tile=tile)
        print(f"Downloading {tile} ...")
        req = urllib.request.Request(url, headers={"User-Agent": "PF8 portfolio notebook (likrentai)"})
        with urllib.request.urlopen(req) as r:
            zpath.write_bytes(r.read())
    with zipfile.ZipFile(zpath) as z:
        names = [n for n in z.namelist() if n.lower().endswith(".tif")]
        tif_path = GHS_DIR / Path(names[0]).name
        if not tif_path.exists():
            tif_path.write_bytes(z.read(names[0]))
    with rasterio.open(tif_path) as src:
        b = src.bounds
        print(f"{tile}: lon {b.left:.2f} to {b.right:.2f}, lat {b.bottom:.2f} to {b.top:.2f}")
        if b.left <= west and b.right >= east and b.bottom <= south and b.top >= north:
            ghs_tif = tif_path
            break
print("Using", ghs_tif)

R13_C11: lon -80.01 to -70.01, lat -40.90 to -30.90
Using data\ghs_built_h\GHS_BUILT_H_ANBH_E2018_GLOBE_R2023A_4326_3ss_V1_0_R13_C11.tif


## 4. Heights: mapped, type rule or grid

In [5]:
import numpy as np
from rasterio.transform import rowcol

with rasterio.open(ghs_tif) as src:
    window = from_bounds(west - 0.01, south - 0.01, east + 0.01, north + 0.01, src.transform)
    ghs = src.read(1, window=window).astype("float32")
    ghs_transform = src.window_transform(window)
    ghs_nodata = src.nodata
if ghs_nodata is not None:
    ghs[ghs == ghs_nodata] = np.nan
ghs[ghs <= 0] = np.nan
print(f"Grid window {ghs.shape}, cell heights {np.nanmin(ghs):.1f} to {np.nanmax(ghs):.1f} m")

centres = keep.to_crs("EPSG:32719").representative_point().to_crs("EPSG:4326")
rows, cols = rowcol(ghs_transform, centres.x.values, centres.y.values)
rows = np.clip(np.asarray(rows), 0, ghs.shape[0] - 1)
cols = np.clip(np.asarray(cols), 0, ghs.shape[1] - 1)
keep = keep.copy()
keep["grid_m"] = ghs[rows, cols]
print(f"Buildings with a grid value: {keep['grid_m'].notna().mean():.1%}")

Grid window (123, 229), cell heights 2.0 to 50.2 m
Buildings with a grid value: 100.0%


How far can the grid be trusted? Compare it with buildings whose height is mapped in OSM. The grid is a cell average, so single tall buildings should come out lower than their mapped height.

In [6]:
mapped = keep[(keep["height_source"] != "none") & keep["grid_m"].notna() & (keep["height_m"] > 0)]
ratio = mapped["height_m"] / mapped["grid_m"]
print(f"{len(mapped):,} mapped buildings with a grid value")
print(f"Mapped height / grid height: median {ratio.median():.2f}, middle half {ratio.quantile(0.25):.2f} to {ratio.quantile(0.75):.2f}")
corr = np.corrcoef(np.log(mapped["height_m"]), np.log(mapped["grid_m"]))[0, 1]
print(f"Correlation of log heights: {corr:.2f}")
bins = [0, 9, 15, 30, 60, 400]
mapped_bins = pd.cut(mapped["height_m"], bins)
print(mapped.groupby(mapped_bins, observed=True)["grid_m"].median().round(1).to_string())

15,052 mapped buildings with a grid value
Mapped height / grid height: median 0.54, middle half 0.33 to 0.91
Correlation of log heights: 0.44
height_m
(0, 9]       11.600000
(9, 15]      13.900000
(15, 30]     19.700001
(30, 60]     23.700001
(60, 400]    26.600000


In [7]:
btype = keep["building"].fillna(keep["building:part"]).fillna("yes")
missing = keep["height_source"].eq("none") | keep["height_m"].fillna(0).le(0)

keep["h"] = keep["height_m"]
keep["est"] = 0
is_house = missing & btype.isin(HOUSE_TYPES)
is_small = missing & btype.isin(SMALL_TYPES)
is_grid = missing & ~is_house & ~is_small & keep["grid_m"].notna()
is_fallback = missing & ~is_house & ~is_small & ~is_grid

keep.loc[is_house, "h"] = HOUSE_FLOORS * LEVEL_HEIGHT_M
keep.loc[is_house, "est"] = 1
keep.loc[is_small, "h"] = SMALL_FLOORS * LEVEL_HEIGHT_M
keep.loc[is_small, "est"] = 1
keep.loc[is_grid, "h"] = keep.loc[is_grid, "grid_m"].clip(lower=MIN_GRID_HEIGHT_M)
keep.loc[is_grid, "est"] = 2
keep.loc[is_fallback, "h"] = FALLBACK_FLOORS * LEVEL_HEIGHT_M
keep.loc[is_fallback, "est"] = 3

keep["mh"] = keep["min_height_m"].fillna(0.0)
bad = keep["mh"] >= keep["h"]
print(f"Parts with base at or above top (fixed by setting base to 0): {bad.sum()}")
keep.loc[bad, "mh"] = 0.0

labels = {0: "mapped in OSM", 1: "type rule", 2: "GHS grid", 3: "fallback"}
summary = keep.groupby("est")["h"].describe()[["count", "50%", "max"]].round(1)
summary.index = [labels[i] for i in summary.index]
print(summary.to_string())

Parts with base at or above top (fixed by setting base to 0): 2
                 count   50%    max
mapped in OSM  15053.0   6.0  300.0
type rule      27595.0   6.0    6.0
GHS grid       29085.0  10.8   49.4
fallback          10.0   6.0    6.0


## 5. Simplify and split into single polygons

Multipolygons become separate rows; only outer rings are kept (holes in building outlines, such as courtyards, are rare and small here).

In [8]:
keep_m = keep.to_crs("EPSG:32719")
if SIMPLIFY_M > 0:
    keep_m["geometry"] = keep_m.geometry.simplify(SIMPLIFY_M, preserve_topology=True)
keep_m = keep_m[~keep_m.geometry.is_empty]
single = keep_m.explode(index_parts=False).to_crs("EPSG:4326")
single = single[single.geom_type == "Polygon"]
print(f"{len(single):,} polygons")

71,773 polygons


## 6. Ground height under each building

Download the Terrarium elevation tiles covering the buildings (once; cached in the folder from Section 0), join them into one grid, and read the ground height at every corner of every outline. Each building gets the **lowest** of its corner heights minus a small sink, so on a slope it never floats.

Terrarium tiles store height in the pixel colours: height = R x 256 + G + B / 256 - 32768 (metres).

In [9]:
import math
import urllib.request
import numpy as np
from PIL import Image


def lonlat_to_tile(lon, lat, z):
    n = 2 ** z
    x = (np.asarray(lon) + 180.0) / 360.0 * n
    lat_r = np.radians(np.asarray(lat))
    y = (1.0 - np.arcsinh(np.tan(lat_r)) / math.pi) / 2.0 * n
    return x, y


west, south, east, north = single.total_bounds
x_min, y_max = lonlat_to_tile(west, south, TERRAIN_ZOOM)
x_max, y_min = lonlat_to_tile(east, north, TERRAIN_ZOOM)
x0 = int(math.floor(x_min))
x1 = int(math.floor(x_max))
y0 = int(math.floor(y_min))
y1 = int(math.floor(y_max))
print(f"Tiles x {x0} to {x1}, y {y0} to {y1}: {(x1 - x0 + 1) * (y1 - y0 + 1)} tiles at zoom {TERRAIN_ZOOM}")

t0 = time.time()
fetched = 0
grid = np.zeros(((y1 - y0 + 1) * 256, (x1 - x0 + 1) * 256), dtype=np.float32)
for ty in range(y0, y1 + 1):
    for tx in range(x0, x1 + 1):
        path = TERRAIN_DIR / f"{tx}_{ty}.png"
        if not path.exists():
            url = TERRAIN_URL.format(z=TERRAIN_ZOOM, x=tx, y=ty)
            req = urllib.request.Request(url, headers={"User-Agent": "PF8 portfolio notebook (likrentai)"})
            with urllib.request.urlopen(req) as r:
                path.write_bytes(r.read())
            fetched += 1
        rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
        height = rgb[:, :, 0] * 256.0 + rgb[:, :, 1] + rgb[:, :, 2] / 256.0 - 32768.0
        r0 = (ty - y0) * 256
        c0 = (tx - x0) * 256
        grid[r0:r0 + 256, c0:c0 + 256] = height
print(f"Downloaded {fetched} new tiles in {time.time() - t0:.0f} s; ground grid {grid.shape}, heights {grid.min():.0f} to {grid.max():.0f} m")

Tiles x 9948 to 9964, y 19611 to 19620: 170 tiles at zoom 15
Downloaded 0 new tiles in 0 s; ground grid (2560, 4352), heights 497 to 899 m


In [10]:
def ground_at(lon, lat):
    tx, ty = lonlat_to_tile(lon, lat, TERRAIN_ZOOM)
    px = (tx - x0) * 256.0 - 0.5
    py = (ty - y0) * 256.0 - 0.5
    px = np.clip(px, 0, grid.shape[1] - 1.001)
    py = np.clip(py, 0, grid.shape[0] - 1.001)
    c = np.floor(px).astype(int)
    r = np.floor(py).astype(int)
    fx = px - c
    fy = py - r
    top = grid[r, c] * (1 - fx) + grid[r, c + 1] * fx
    bottom = grid[r + 1, c] * (1 - fx) + grid[r + 1, c + 1] * fx
    return top * (1 - fy) + bottom * fy


ring_lons = []
ring_lats = []
ring_owner = []
for i, geom in enumerate(single.geometry):
    xy = np.asarray(geom.exterior.coords)[:-1]
    ring_lons.append(xy[:, 0])
    ring_lats.append(xy[:, 1])
    ring_owner.append(np.full(len(xy), i))
all_lon = np.concatenate(ring_lons)
all_lat = np.concatenate(ring_lats)
owner = np.concatenate(ring_owner)

corner_ground = ground_at(all_lon, all_lat)
lowest = np.full(len(single), np.inf)
np.minimum.at(lowest, owner, corner_ground)
single = single.copy()
single["g"] = lowest - SINK_M
print(f"Ground under buildings: {single['g'].min():.0f} to {single['g'].max():.0f} m (median {single['g'].median():.0f} m)")

Ground under buildings: 504 to 887 m (median 606 m)


## 7. Write the compact JSON

Format, chosen to keep the file small:

```
{"names": [...], "cols": ["h", "mh", "est", "name", "g", "ring"], "rows": [[h, mh, est, nameIndex, ground, [lon, lat, lon, lat, ...]], ...]}
```

`nameIndex` is -1 when the building has no name. `ground` is the ground height in metres above sea level (Section 6); `est` is the height source (0 mapped, 1 type rule, 2 grid, 3 fallback); the page places the building's base at ground + base height. The ring is a flat list of coordinates.

In [11]:
import json

names = sorted(single["name"].dropna().unique().tolist())
name_index = {n: i for i, n in enumerate(names)}

rows = []
for h, mh, est, name, g, geom in zip(single["h"], single["mh"], single["est"], single["name"], single["g"], single.geometry):
    ring = []
    for x, y in list(geom.exterior.coords)[:-1]:
        ring.append(round(x, DECIMALS))
        ring.append(round(y, DECIMALS))
    ni = name_index.get(name, -1) if isinstance(name, str) else -1
    rows.append([round(float(h), 1), round(float(mh), 1), int(est), ni, round(float(g), 1), ring])

data = {"names": names, "cols": ["h", "mh", "est", "name", "g", "ring"], "rows": rows}
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, separators=(",", ":"), ensure_ascii=False)
print(f"Wrote {len(rows):,} polygons, {len(names):,} names to {OUT_FILE} ({OUT_FILE.stat().st_size / 1e6:.1f} MB)")

Wrote 71,773 polygons, 4,690 names to docs\data\buildings.json (11.8 MB)


___
**END**